In [ ]:
import yaml
import numpy as np
import prettytable as pt

In [ ]:
with open('telescope_params.yaml', 'r') as file:
    params = yaml.safe_load(file)

In [ ]:
def logit(x):
    return np.log(x / (1 - x))


def t_exp_desi(m_i,p_success):
    return 10**(-15.38 + 0.8*m_i +0.67*logit(p_success))

def sec_to_dark_years(sec):
    return (2/3)*sec/(60*60*8*365)

def scaled_exp_time(m_i, p_success, telescope_area, desi_area=8.66):
    return sec_to_dark_years(t_exp_desi(m_i, p_success)) * (desi_area / telescope_area)

In [ ]:
def field_limited_time(fov, telescope_area, m_i, p_success, desi_area=8.66):
    if fov>=3:
        nfields = 4
    elif fov<=1:
        nfields = 15
    else:
        nfields = 6
    return nfields * scaled_exp_time(m_i, p_success, telescope_area, desi_area)

def multiplex_limited_time(multiplex, telescope_area, m_i, p_success, total_objects, desi_area=8.66):
    return (total_objects / multiplex) * scaled_exp_time(m_i, p_success, telescope_area, desi_area)

def area_limited_time(fov, telescope_area, m_i, p_success, total_area, desi_area=8.66):
    return (total_area / fov) * scaled_exp_time(m_i, p_success, telescope_area, desi_area)

def total_time(fov, telescope_area, multiplex, m_i, p_success, total_objects, total_area, desi_area=8.66):
    field_time = field_limited_time(fov, telescope_area, m_i, p_success, desi_area)
    multiplex_time = multiplex_limited_time(multiplex, telescope_area, m_i, p_success, total_objects, desi_area)
    area_time = area_limited_time(fov, telescope_area, m_i, p_success, total_area, desi_area)
    time = max(field_time, multiplex_time, area_time)
    limit = ["field", "multiplex", "FoV"][np.argmax([field_time, multiplex_time, area_time])]
    return time , limit

In [ ]:
table = pt.PrettyTable()
table.field_names = ["Telescope", "Area (m^2)", "FOV (deg^2)", "Multiplex", "Total Time (years)", "Limiting Factor"]
m_i = 25.3
p_success = 0.9
total_objects = 20000
total_area = 1.5 # in deg^2 equivalent to 15 fields of 10 arc min radius.


for telescope, param in params.items():
    time,limit = total_time(fov=param['fov'], telescope_area=param['area'], multiplex=param['multiplex'], m_i=m_i, p_success=p_success, total_objects=total_objects, total_area=total_area, desi_area=8.66)
    table.add_row([telescope, param['area'], param['fov'], param['multiplex'], f"{time:.2f}", limit])
print(table)